# TICKET MJD-602

De: Lucía Gómez · Administración y finanzas
En el MJD-204 cuadrasteis marzo de 2025 con mi hoja de facturación. Quiero lo mismo para todos los
meses, de enero de 2024 a septiembre de 2026. Si alguno no cuadra, quiero saber exactamente qué
pedidos lo explican.

CRITERIOS
- La hoja facturacion_mensual_2024-2026.xlsx leída con pandas y comparada mes a mes con gold.
- Lista de los meses que no cuadran, con la diferencia.
- Los pedidos que explican cada diferencia.

In [17]:
import os, sys
from pathlib import Path

RAIZ = Path.home() / "master-data-ia"
os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))

In [11]:
import pandas as pd
pd.set_option("display.float_format", "{:,.2f}".format)
conta = pd.read_excel("datos/contabilidad/facturacion_mensual_2024-2026.xlsx", skiprows=3)
meses = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio",
    "agosto", "septiembre", "octubre", "noviembre", "diciembre"]
conta["mes"] = conta["Mes"].map({m: i for i, m in enumerate(meses, start=1)})
assert conta["mes"].notna().all(), "hay algún mes mal escrito"
contab = (
    conta.groupby(["Año", "mes"])["Base imponible (€)"].sum()
    .rename_axis(["anio", "mes"]).rename("contabilidad")
)
print(len(contab))
contab.head(3)

33


anio  mes
2024  1     811,290.05
      2     805,880.38
      3     949,850.27
Name: contabilidad, dtype: float64

Gold, con el criterio del puente del Volumen 02: pedidos entregados o con devolución parcial, por mes de en‐
trega, con los portes. Es la comprobación 7 de 04_comprobar_gold.sql , ahora para todos los meses:

In [13]:
import duckdb
con = duckdb.connect("db/marejada.duckdb", read_only=True)
gold = con.sql("""
SELECT d.anio, d.mes, sum(f.importe_neto + f.gastos_envio) AS gold
FROM gold.fact_pedidos AS f
JOIN gold.dim_fecha AS d ON d.fecha_key = f.fecha_entrega_key
WHERE f.estado IN ('Entregado', 'Devuelto parcial')
GROUP BY d.anio, d.mes
""").df().set_index(["anio", "mes"])["gold"]
cuadre = pd.concat([contab, gold], axis=1)
cuadre["dif"] = (cuadre["contabilidad"] - cuadre["gold"]).round(2)
no_cuadran = cuadre[cuadre["dif"].abs() > 0.01]
print(len(cuadre), len(no_cuadran))
no_cuadran

33 8


contabilidad         gold      dif
anio mes                                    
2024 7    1,433,547.21 1,433,231.86   315.35
     8    1,394,522.90 1,394,451.72    71.18
     10   1,062,285.93 1,062,172.99   112.94
     12   1,224,133.48 1,223,978.80   154.68
2025 1      991,585.53   991,137.19   448.34
     5    1,235,046.85 1,231,799.73 3,247.12
     7    1,676,382.85 1,675,403.02   979.83
     8    1,651,705.30 1,651,315.28   390.02

Ocho meses no cuadran, todos por poco, y siempre con contabilidad por encima de gold. Algo que contabili‐
dad ha facturado en ese mes y gold no tiene en ese mes. ¿Qué pedidos entregados no tienen mes de entrega
en gold? Los de la fecha imposible, que silver dejó a NULL (Volumen 02, MJD-210). Su fecha original está en
bronze:

In [14]:
raras = con.sql("""
SELECT f.pedido_id, b.fecha_entrega AS fecha_en_erp,
round(f.importe_neto + f.gastos_envio, 2) AS importe
FROM gold.fact_pedidos AS f
JOIN bronze.pedidos AS b USING (pedido_id)
WHERE f.estado IN ('Entregado', 'Devuelto parcial')
AND f.fecha_entrega_key IS NULL
""").df()
con.close()
raras["fecha_en_erp"] = pd.to_datetime(raras["fecha_en_erp"])
raras["anio"] = raras["fecha_en_erp"].dt.year
raras["mes"] = raras["fecha_en_erp"].dt.month
print(len(raras))
raras["anio"].value_counts().sort_index()

23


anio
1900    9
2023    4
2024    4
2025    6
Name: count, dtype: int64

Los 23 pedidos de fecha imposible: 9 con el 1 de enero de 1900 y 14 con el año equivocado. Si contabilidad
facturó con la fecha que tenía el ERP, los que caen dentro del periodo de la hoja (2024 a 2026) explicarían
las diferencias. Comprobación: sumarlos por ese mes y compararlo con la diferencia.

In [15]:
explican = raras[raras["anio"].between(2024, 2026)].groupby(["anio", "mes"])["importe"].sum()
comparacion = pd.concat([no_cuadran["dif"], explican.rename("pedidos_raros")], axis=1)
comparacion["resto"] = (comparacion["dif"] - comparacion["pedidos_raros"]).round(2)
comparacion

dif  pedidos_raros  resto
anio mes                               
2024 7     315.35         315.34   0.01
     8      71.18          71.18   0.00
     10    112.94         112.94   0.00
     12    154.68         154.67   0.01
2025 1     448.34         448.34   0.00
     5   3,247.12       3,247.13  -0.01
     7     979.83         979.83   0.00
     8     390.02         390.03  -0.01

Cuadra: cada diferencia es exactamente la suma de los pedidos que el ERP tenía fechados en ese mes, con
algún céntimo de redondeo (las diferencias se redondearon mes a mes; los importes, pedido a pedido). Con‐
tabilidad facturó con la fecha errónea, un año antes de la entrega real.

In [16]:
lista = raras[raras["anio"].between(2024, 2026)].sort_values("fecha_en_erp")
print(len(lista), round(lista["importe"].sum(), 2))
lista[["pedido_id", "fecha_en_erp", "importe"]]

10 5719.46


,pedido_id,fecha_en_erp,importe
6,2025013364,2024-07-15,315.34
9,2025015502,2024-08-07,71.18
10,2025021740,2024-10-13,112.94
11,2025026215,2024-12-10,154.67
12,2026000597,2025-01-12,448.34
15,2026009133,2025-05-04,534.02
16,2026010798,2025-05-21,"2,713.11"
18,2026015558,2025-07-07,485.69
20,2026018492,2025-07-31,494.14
21,2026018807,2025-08-05,390.03
